# Clase 5 — Memoria, contexto y estado

Hasta ahora cada consulta comenzó desde cero. Una conversación real incluye referencias como mi ticket, ese error o lo de ayer.

La memoria puede ayudar, pero también puede mezclar usuarios, aumentar el contexto y conservar datos que no necesitamos.

## Objetivos

- Diferenciar historial, memoria de trabajo, perfil y estado.
- Relacionar memoria con ventana de contexto y atención.
- Incorporar memoria limitada a ambos agentes.
- Comparar respuestas con y sin contexto.
- Probar aislamiento, resumen y derecho al olvido.

---
## 1. Cuatro conceptos que suelen confundirse

| Componente | Pregunta que responde | Ejemplo |
|---|---|---|
| Historial | ¿Qué se dijo? | últimos mensajes |
| Memoria de trabajo | ¿Qué necesito ahora? | ticket activo T005 |
| Perfil | ¿Qué dato estable fue confirmado? | canal preferido |
| Estado | ¿En qué paso está el proceso? | esperando número de ticket |

El LLM no recuerda una ejecución anterior por sí mismo. El programa decide qué información vuelve a incluir en la siguiente consulta.

In [ ]:
sesion = {
    "usuario_id":"U01",
    "historial":[],
    "trabajo":{},
    "perfil":{},
    "estado":"inicio",
}
sesion

---
## 2. Una memoria con límites

Usaremos un diccionario de sesiones para evitar que dos usuarios compartan historial. Cada historial conservará solo seis mensajes.

In [ ]:
class MemoriaSesiones:
    def __init__(self, limite_mensajes=6):
        self.limite = limite_mensajes
        self.sesiones = {}

    def obtener(self, usuario_id):
        if usuario_id not in self.sesiones:
            self.sesiones[usuario_id] = {
                "historial":[], "trabajo":{},
                "perfil":{}, "estado":"inicio",
            }
        return self.sesiones[usuario_id]

    def agregar_mensaje(self, usuario_id, rol, contenido):
        sesion = self.obtener(usuario_id)
        sesion["historial"].append({"rol":rol, "contenido":contenido})
        sesion["historial"] = sesion["historial"][-self.limite:]

    def olvidar(self, usuario_id):
        return self.sesiones.pop(usuario_id, None) is not None

memoria = MemoriaSesiones()
memoria.agregar_mensaje("U01","persona","No puedo exportar")
memoria.agregar_mensaje("U02","persona","No puedo ingresar")
memoria.sesiones

### Cómo leer el resultado

U01 y U02 tienen estructuras independientes. La clave usuario_id no es una contraseña ni un dato que el LLM deba inventar: la entrega la aplicación autenticada.

---
## 3. Memoria de trabajo: completar referencias

Guardaremos solamente el ticket activo y el servicio mencionado. No intentaremos extraer todos los datos personales de una frase.

In [ ]:
def actualizar_trabajo(memoria, usuario_id, ticket_id=None, servicio=None):
    trabajo = memoria.obtener(usuario_id)["trabajo"]
    if ticket_id:
        trabajo["ticket_activo"] = ticket_id.upper()
    if servicio:
        trabajo["servicio_activo"] = servicio.lower()
    return trabajo

actualizar_trabajo(memoria, "U01", ticket_id="T005", servicio="reportes")
memoria.obtener("U01")

In [ ]:
def resolver_referencia(memoria, usuario_id, consulta):
    trabajo = memoria.obtener(usuario_id)["trabajo"]
    texto = consulta
    if "mi ticket" in consulta.lower() and "ticket_activo" in trabajo:
        texto += f" [ticket_activo={trabajo['ticket_activo']}]"
    if "ese servicio" in consulta.lower() and "servicio_activo" in trabajo:
        texto += f" [servicio_activo={trabajo['servicio_activo']}]"
    return texto

print(resolver_referencia(memoria, "U01", "¿Cómo sigue mi ticket?"))
print(resolver_referencia(memoria, "U02", "¿Cómo sigue mi ticket?"))

---
## 4. Ventana de contexto y atención

Un Transformer relaciona tokens dentro de una ventana limitada. Agregar historial puede resolver referencias, pero también:

- ocupa espacio;
- aumenta tiempo de inferencia;
- introduce mensajes irrelevantes;
- puede arrastrar instrucciones maliciosas;
- puede exponer datos.

Más contexto no siempre significa mejor contexto.

In [ ]:
def contexto_para_modelo(memoria, usuario_id):
    sesion = memoria.obtener(usuario_id)
    lineas = []
    for mensaje in sesion["historial"]:
        lineas.append(f"{mensaje['rol']}: {mensaje['contenido']}")
    if sesion["trabajo"]:
        lineas.append(f"estado_confirmado: {sesion['trabajo']}")
    return "\n".join(lineas)

contexto = contexto_para_modelo(memoria, "U01")
print(contexto)
print("\nCaracteres enviados:", len(contexto))
print("Tokens aproximados:", round(len(contexto) / 4))

### La atención no es memoria permanente

La atención calcula relaciones entre elementos del contexto que enviamos en esa llamada. La memoria del agente está en nuestras estructuras; el Transformer solo ve la porción que incluimos.

---
## 5. Agente A y agente B con el mismo estado

El agente A usa el estado para completar reglas. El agente B recibe una versión textual del mismo estado. Así la comparación sigue siendo justa.

In [ ]:
def agente_reglas_con_memoria(memoria, usuario_id, consulta):
    entrada = resolver_referencia(memoria, usuario_id, consulta)
    memoria.agregar_mensaje(usuario_id, "persona", consulta)
    if "ticket_activo=" in entrada:
        respuesta = "Consultaré el ticket confirmado en la sesión."
        decision = "consultar_ticket"
    else:
        respuesta = "Necesito el número de ticket."
        decision = "pedir_dato"
    memoria.agregar_mensaje(usuario_id, "agente", respuesta)
    return {"decision":decision, "respuesta":respuesta,
            "contexto_usado":entrada}

agente_reglas_con_memoria(memoria, "U01", "¿Cómo sigue mi ticket?")

In [ ]:
def preparar_prompt_llm(memoria, usuario_id, consulta):
    contexto = contexto_para_modelo(memoria, usuario_id)
    return f"""CONTEXTO CONFIRMADO:
{contexto}

CONSULTA ACTUAL:
{consulta}

Usá solo el contexto confirmado. Si falta el ticket, pedilo.
No mezcles información de otras sesiones."""

print(preparar_prompt_llm(memoria, "U01", "¿Cómo sigue mi ticket?"))

---
## 6. Resumir sin inventar

Cuando el historial crece, podemos conservar hechos confirmados y eliminar charla repetida. En esta clase el resumen es determinista: no pedimos al LLM que decida qué dato sensible guardar.

In [ ]:
def resumir_sesion(memoria, usuario_id):
    sesion = memoria.obtener(usuario_id)
    return {
        "usuario_id":usuario_id,
        "ticket_activo":sesion["trabajo"].get("ticket_activo"),
        "servicio_activo":sesion["trabajo"].get("servicio_activo"),
        "estado":sesion["estado"],
        "mensajes_conservados":len(sesion["historial"]),
    }

resumir_sesion(memoria, "U01")

---
## 7. Contaminación entre usuarios

Un error grave sería reutilizar el historial de U01 al responder a U02. La prueba debe verificar ausencia, no solo que el programa no falle.

In [ ]:
def prueba_aislamiento(memoria):
    memoria.agregar_mensaje("TEST_A","persona","Mi ticket es T999")
    actualizar_trabajo(memoria, "TEST_A", ticket_id="T999")
    memoria.agregar_mensaje("TEST_B","persona","Hola")
    contexto_b = contexto_para_modelo(memoria, "TEST_B")
    return {
        "contexto_b":contexto_b,
        "contiene_dato_ajeno":"T999" in contexto_b,
        "aprobada":"T999" not in contexto_b,
    }

prueba_aislamiento(memoria)

---
## 📝 Actividad 1 — Completar el estado conversacional

Agregá un estado esperando_servicio. Si el agente preguntó qué servicio falla, el siguiente mensaje debe completar servicio_activo y volver a estado inicio.

In [ ]:
def procesar_estado(memoria, usuario_id, mensaje):
    sesion = memoria.obtener(usuario_id)
    # TODO: si estado == "esperando_servicio", guardar mensaje como servicio
    # TODO: volver a "inicio"
    # TODO: si el mensaje dice "falla", pasar a "esperando_servicio"
    return sesion

procesar_estado(memoria, "U03", "falla una aplicación")

---
## 📝 Actividad 2 — Probar el límite de contexto

Agregá diez mensajes a una sesión. Comprobá que solo queden seis y compará el contexto antes y después del resumen.

In [ ]:
for numero in range(10):
    memoria.agregar_mensaje("U04", "persona", f"mensaje {numero}")

historial_u04 = memoria.obtener("U04")["historial"]
print("Mensajes conservados:", len(historial_u04))
print(historial_u04)

---
## 📝 Actividad 3 — Derecho al olvido

Ejecutá olvidar y verificá tres condiciones: no queda historial, no queda perfil y una nueva sesión comienza vacía.

In [ ]:
memoria.obtener("U05")["perfil"]["canal"] = "correo"
memoria.agregar_mensaje("U05","persona","consulta privada")

# TODO: llamar memoria.olvidar("U05")
# TODO: comprobar que obtener("U05") crea una sesión vacía
verificacion_olvido = {"historial_vacio":False,
                       "perfil_vacio":False,
                       "trabajo_vacio":False}
verificacion_olvido

---
## ✅ Resumen

La memoria quedó fuera del LLM y bajo control del programa. Distinguimos historial, trabajo, perfil y estado; limitamos contexto; aislamos usuarios y diseñamos eliminación.

En la Clase 6 agregaremos conocimiento externo. El desafío será responder con evidencia recuperada y reconocer cuándo esa evidencia no existe.